Письмові відповіді
1. Чи можете пояснити, чому заповнення пропуску середнім (а не медіаною) може згодом спотворити стандартне відхилення, яке рахуватимемо в наступних лекціях?
Заповнення пропусків середнім значенням може зменшити варіативність набору даних, оскільки пропущені значення замінюються одним і тим самим значенням, близьким до центру розподілу. Через це стандартне відхилення може стати штучно меншим і не повністю відображати реальний розкид даних. У нашому випадку медіана є доречнішою, оскільки вона менш чутлива до нетипових значень і краще підходить для невеликого набору даних.

2. Чи розумієте різницю між duplicated() без subset і з subset — чому результат може відрізнятись?
duplicated() без параметра subset перевіряє повний збіг усіх значень у рядках. Якщо використати subset, наприклад subset=["id"], то перевірка дублювання виконується тільки за вказаними стовпцями. Тому результат може відрізнятися: два рядки можуть мати різні значення в інших стовпцях, але вважатися дубльованими, якщо мають однаковий унікальний ідентифікатор.

3. Чи можете пояснити, чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення?
Автоматичне видалення всіх значень поза межами IQR не завжди правильне, тому що такі значення не обов'язково є помилками. Вони можуть бути реальними та важливими спостереженнями. Тому перед видаленням потрібно перевірити причину появи такого значення та врахувати предметну область. У нашому випадку ціна 6400 грн є підозрілою, оскільки вона у 10 разів перевищує базову ціну 640 грн, тому ми вважаємо її помилкою вводу із зайвим нулем і виправляємо на 640 грн.

In [31]:
print(orders.dtypes)


id             int64
місто            str
товар            str
ціна         float64
кількість    float64
dtype: object


In [30]:
print("Міста:", orders["місто"].unique())


Міста: <StringArray>
['одеса']
Length: 1, dtype: str


In [29]:
print("Дублікати:", orders.duplicated().sum())


Дублікати: 0


In [28]:
print("Пропуски:")
print(orders.isna().sum())


Пропуски:
id           0
місто        0
товар        0
ціна         0
кількість    0
dtype: int64


In [27]:
orders


,id,місто,товар,ціна,кількість
0,1,одеса,Товар,640.0,5.0
1,2,одеса,Товар,640.0,4.0
2,3,одеса,Товар,640.0,4.5
3,4,одеса,Товар,640.0,6.0
4,5,одеса,Товар,640.0,4.5
5,6,одеса,Товар,640.0,5.0
6,7,одеса,Товар,640.0,3.0


In [26]:
orders.loc[orders["ціна"] == 6400, "ціна"] = 640


Рішення щодо викиду

За критерієм IQR значення 6400 грн є викидом, оскільки воно перевищує верхню межу IQR. У цьому наборі я вважаю 6400 грн помилкою вводу, а не реальним великим замовленням. Базова ціна товару становить 640 грн, а 6400 грн рівно у 10 разів більша за неї, тому найбільш логічним поясненням є випадково доданий нуль. Тому це значення доцільно виправити з 6400 грн на 640 грн, а не залишати як реальне спостереження.

In [25]:
print("Викиди:")
print(outliers)


Викиди:
   id  місто  товар    ціна  кількість
6   7  одеса  Товар  6400.0        3.0


In [24]:
outliers = orders[
    (orders["ціна"] < lower) |
    (orders["ціна"] > upper)
]

outliers


,id,місто,товар,ціна,кількість
6,7,одеса,Товар,6400.0,3.0


In [23]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Нижня межа:", lower)
print("Верхня межа:", upper)


Нижня межа: 640.0
Верхня межа: 640.0


In [22]:
iqr = q3 - q1

print("IQR =", iqr)


IQR = 0.0


In [21]:
q1 = orders["ціна"].quantile(0.25)
q3 = orders["ціна"].quantile(0.75)

print("Q1 =", q1)
print("Q3 =", q3)


Q1 = 640.0
Q3 = 640.0


In [20]:
orders


,id,місто,товар,ціна,кількість
0,1,одеса,Товар,640.0,5.0
1,2,одеса,Товар,640.0,4.0
2,3,одеса,Товар,640.0,4.5
3,4,одеса,Товар,640.0,6.0
4,5,одеса,Товар,640.0,4.5
5,6,одеса,Товар,640.0,5.0
6,7,одеса,Товар,6400.0,3.0


In [19]:
print(orders["місто"].unique())


<StringArray>
['одеса']
Length: 1, dtype: str


In [18]:
orders["місто"] = (
    orders["місто"]
    .str.strip()
    .str.lower()
)


In [17]:
print(orders.dtypes)


id             int64
місто            str
товар            str
ціна         float64
кількість    float64
dtype: object


In [16]:
orders["ціна"] = (
    orders["ціна"]
    .astype(str)
    .str.replace(" грн", "", regex=False)
    .astype(float)
)


In [15]:
print(orders.dtypes)


id             int64
місто            str
товар            str
ціна          object
кількість    float64
dtype: object


In [14]:
print("Кількість рядків після видалення дубліката:", len(orders))


Кількість рядків після видалення дубліката: 7


In [13]:
print(orders)


   id    місто  товар     ціна  кількість
0   1    Одеса  Товар      640        5.0
1   2   Одеса   Товар  640 грн        4.0
2   3    ОДЕСА  Товар  640 грн        4.5
3   4    Одеса  Товар      640        6.0
4   5    ОДЕСА  Товар      640        4.5
5   6    Одеса  Товар      640        5.0
6   7    Одеса  Товар     6400        3.0


In [12]:
orders = orders.drop_duplicates(subset=["id"])


In [11]:
print(
    "Кількість дублікатів за id:",
    orders.duplicated(subset=["id"]).sum()
)


Кількість дублікатів за id: 1


In [10]:
print("Дублікати за id:")
print(orders.duplicated(subset=["id"]))


Дублікати за id:
0    False
1    False
2    False
3    False
4    False
5    False
6    False
7     True
dtype: bool


In [9]:
print("Кількість дублікатів:", orders.duplicated().sum())


Кількість дублікатів: 1


In [8]:
print("Дублікати без subset:")
print(orders.duplicated())


Дублікати без subset:
0    False
1    False
2    False
3    False
4    False
5    False
6    False
7     True
dtype: bool


In [7]:
print(orders["кількість"])


0    5.0
1    4.0
2    4.5
3    6.0
4    4.5
5    5.0
6    3.0
7    3.0
Name: кількість, dtype: float64


In [6]:
orders["кількість"] = orders["кількість"].fillna(
    orders["кількість"].median()
)


In [5]:
print(orders["кількість"].median())


4.5


Завдання 2. Виявлення та заповнення пропусків

In [4]:
print(orders.isna().sum())


id           0
місто        0
товар        0
ціна         0
кількість    2
dtype: int64


In [3]:
print(orders.shape)


(8, 5)


In [2]:
orders = pd.DataFrame([
    {"id": 1, "місто": "Одеса",   "товар": "Товар", "ціна": 640,       "кількість": 5},
    {"id": 2, "місто": " Одеса ", "товар": "Товар", "ціна": "640 грн", "кількість": 4},
    {"id": 3, "місто": "ОДЕСА",  "товар": "Товар", "ціна": "640 грн", "кількість": np.nan},
    {"id": 4, "місто": "Одеса",   "товар": "Товар", "ціна": 640,       "кількість": 6},
    {"id": 5, "місто": "ОДЕСА",  "товар": "Товар", "ціна": 640,       "кількість": np.nan},
    {"id": 6, "місто": "Одеса",   "товар": "Товар", "ціна": 640,       "кількість": 5},
    {"id": 7, "місто": "Одеса",   "товар": "Товар", "ціна": 6400,      "кількість": 3},
    {"id": 7, "місто": "Одеса",   "товар": "Товар", "ціна": 6400,      "кількість": 3},
])

orders


,id,місто,товар,ціна,кількість
0,1,Одеса,Товар,640,5.0
1,2,Одеса,Товар,640 грн,4.0
2,3,ОДЕСА,Товар,640 грн,NaN
3,4,Одеса,Товар,640,6.0
4,5,ОДЕСА,Товар,640,NaN
5,6,Одеса,Товар,640,5.0
6,7,Одеса,Товар,6400,3.0
7,7,Одеса,Товар,6400,3.0


In [1]:
import numpy as np
import pandas as pd
